# POC-21 Phase 3 Post #3

## Analytical Track Development and Validation

**Approved Track:** Track A - Comparative Intelligence  
**Canonical Dataset:** `data/canonical/intelligence_data.csv`  
**Data Version:** 1.1.0  
**Method Version:** 1.0.0


In [3]:
from pathlib import Path
import json
import pandas as pd

# Locate the actual repository root.
current = Path.cwd().resolve()
PROJECT_ROOT = None

for candidate in [current, *current.parents]:
    if (candidate / "data" / "canonical" / "intelligence_data.csv").is_file():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate the canonical dataset from: " + str(current)
    )

CANONICAL_PATH = PROJECT_ROOT / "data" / "canonical" / "intelligence_data.csv"
OUTPUT_DIR = PROJECT_ROOT / "data-science" / "outputs"

print("Project root:", PROJECT_ROOT)
print("Canonical dataset:", CANONICAL_PATH)
print("Dataset exists:", CANONICAL_PATH.exists())
print("Output directory:", OUTPUT_DIR)


Project root: C:\Users\SURESH\POC-21-Downtime-Root-Cause-Explorer-Niranjana-Phase3
Canonical dataset: C:\Users\SURESH\POC-21-Downtime-Root-Cause-Explorer-Niranjana-Phase3\data\canonical\intelligence_data.csv
Dataset exists: True
Output directory: C:\Users\SURESH\POC-21-Downtime-Root-Cause-Explorer-Niranjana-Phase3\data-science\outputs


In [4]:
if not CANONICAL_PATH.exists():
    raise FileNotFoundError(f'Canonical dataset not found: {CANONICAL_PATH}')

df = pd.read_csv(CANONICAL_PATH)

print('Measurement rows:', len(df))
print('Columns:', list(df.columns))
print('Data versions:', sorted(df['data_version'].dropna().astype(str).unique()))
print('Distinct source incidents:', df['source_record_id'].nunique())

df.head()


Measurement rows: 60
Columns: ['record_id', 'record_type', 'observed_at', 'entity_id', 'related_entity_id', 'entity_name', 'category', 'subcategory', 'status', 'stage', 'metric_name', 'metric_value', 'metric_unit', 'text_value', 'latitude', 'longitude', 'source_name', 'source_record_id', 'is_synthetic', 'data_version', 'shift', 'severity']
Data versions: ['1.1.0']
Distinct source incidents: 30


,record_id,record_type,observed_at,entity_id,related_entity_id,entity_name,category,subcategory,status,stage,...,metric_unit,text_value,latitude,longitude,source_name,source_record_id,is_synthetic,data_version,shift,severity
0,POC21-INC001-DOWNTIME,measurement,2026-01-05T00:00:00Z,MCH-01,NaN,Machine MCH-01,mechanical,bearing_failure,closed,NaN,...,minutes,Bearing wear caused machine stoppage,NaN,NaN,synthetic_downtime_incidents_csv,INC001,True,1.1.0,morning,high
1,POC21-INC001-FREQUENCY,measurement,2026-01-05T00:00:00Z,MCH-01,NaN,Machine MCH-01,mechanical,bearing_failure,closed,NaN,...,count,Bearing wear caused machine stoppage,NaN,NaN,synthetic_downtime_incidents_csv,INC001,True,1.1.0,morning,high
2,POC21-INC002-DOWNTIME,measurement,2026-01-06T00:00:00Z,MCH-02,NaN,Machine MCH-02,electrical,motor_failure,closed,NaN,...,minutes,Motor overheating interrupted production,NaN,NaN,synthetic_downtime_incidents_csv,INC002,True,1.1.0,evening,medium
3,POC21-INC002-FREQUENCY,measurement,2026-01-06T00:00:00Z,MCH-02,NaN,Machine MCH-02,electrical,motor_failure,closed,NaN,...,count,Motor overheating interrupted production,NaN,NaN,synthetic_downtime_incidents_csv,INC002,True,1.1.0,evening,medium
4,POC21-INC003-DOWNTIME,measurement,2026-01-07T00:00:00Z,MCH-03,NaN,Machine MCH-03,operational,operator_error,closed,NaN,...,minutes,Incorrect machine setup,NaN,NaN,synthetic_downtime_incidents_csv,INC003,True,1.1.0,night,low


## Input Validation

The analytical workflow uses only the approved canonical dataset and verifies the required fields before reconstruction.


In [5]:
required_fields = [
    'source_record_id',
    'category',
    'entity_id',
    'shift',
    'metric_name',
    'metric_value',
    'data_version',
    'is_synthetic'
]

missing_fields = [field for field in required_fields if field not in df.columns]

if missing_fields:
    raise ValueError(f'Missing required fields: {missing_fields}')

print('Required field validation: PASS')
print('Expected data version: 1.1.0')
print('Actual data versions:', sorted(df['data_version'].dropna().astype(str).unique()))
print('All records synthetic:', bool(df['is_synthetic'].astype(bool).all()))


Required field validation: PASS
Expected data version: 1.1.0
Actual data versions: ['1.1.0']
All records synthetic: True


## Incident Reconstruction

Each source incident is reconstructed by pairing `downtime_minutes` and `failure_frequency` using `source_record_id`.


In [6]:
working = df.copy()
working['metric_value'] = pd.to_numeric(working['metric_value'], errors='coerce')

invalid_numeric_rows = int(working['metric_value'].isna().sum())

pivot = working.pivot_table(
    index=['source_record_id', 'category', 'entity_id', 'shift'],
    columns='metric_name',
    values='metric_value',
    aggfunc='first'
).reset_index()

required_metrics = ['downtime_minutes', 'failure_frequency']

for metric in required_metrics:
    if metric not in pivot.columns:
        pivot[metric] = pd.NA

complete_incidents = pivot.dropna(subset=required_metrics).copy()

print('Invalid numeric rows:', invalid_numeric_rows)
print('Reconstructed incidents:', len(pivot))
print('Complete incidents:', len(complete_incidents))

complete_incidents.head()


Invalid numeric rows: 0
Reconstructed incidents: 30
Complete incidents: 30


metric_name,source_record_id,category,entity_id,shift,downtime_minutes,failure_frequency
0,INC001,mechanical,MCH-01,morning,120,3
1,INC002,electrical,MCH-02,evening,90,2
2,INC003,operational,MCH-03,night,45,1
3,INC004,mechanical,MCH-01,morning,75,2
4,INC005,electrical,MCH-04,evening,150,4


## Baseline and Comparative Intelligence

The baseline is descriptive group comparison using incident count, total downtime, average downtime, total failure frequency, and average failure frequency.


In [7]:
def calculate_group_results(data, group_field):
    result = (
        data.groupby(group_field, dropna=False)
        .agg(
            incident_count=('source_record_id', 'nunique'),
            total_downtime=('downtime_minutes', 'sum'),
            average_downtime=('downtime_minutes', 'mean'),
            total_failure_frequency=('failure_frequency', 'sum'),
            average_failure_frequency=('failure_frequency', 'mean')
        )
        .reset_index()
    )

    result[group_field] = result[group_field].fillna('(blank)').astype(str)

    result = result.sort_values(
        by=['total_downtime', 'incident_count', group_field],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    result['rank_by_total_downtime'] = range(1, len(result) + 1)

    return result

category_results = calculate_group_results(complete_incidents, 'category')
machine_results = calculate_group_results(complete_incidents, 'entity_id')
shift_results = calculate_group_results(complete_incidents, 'shift')

print('Top category:')
display(category_results.head())

print('Top machine:')
display(machine_results.head())

print('Top shift:')
display(shift_results.head())


Top category:


,category,incident_count,total_downtime,average_downtime,total_failure_frequency,average_failure_frequency,rank_by_total_downtime
0,mechanical,10,1265,126.500000,28,2.800000,1
1,electrical,9,950,105.555556,23,2.555556,2
2,maintenance,5,360,72.000000,8,1.600000,3
3,operational,6,220,36.666667,6,1.000000,4


Top machine:


,entity_id,incident_count,total_downtime,average_downtime,total_failure_frequency,average_failure_frequency,rank_by_total_downtime
0,MCH-02,8,850,106.250000,20,2.500000,1
1,MCH-04,7,700,100.000000,17,2.428571,2
2,MCH-01,8,635,79.375000,15,1.875000,3
3,MCH-03,7,610,87.142857,13,1.857143,4


Top shift:


,shift,incident_count,total_downtime,average_downtime,total_failure_frequency,average_failure_frequency,rank_by_total_downtime
0,morning,11,1060,96.363636,26,2.363636,1
1,evening,10,935,93.500000,22,2.200000,2
2,night,9,800,88.888889,17,1.888889,3


## Validation and Limitations

Validation confirms row counts, incident reconstruction, complete metric pairing, version consistency, synthetic-data status, group coverage, and deterministic ranking. Results remain descriptive because the dataset contains synthetic incidents and small observed groups.


In [8]:
validation_summary = {
    'measurement_rows': int(len(df)),
    'distinct_source_incidents': int(df['source_record_id'].nunique()),
    'reconstructed_incidents': int(len(pivot)),
    'complete_incidents': int(len(complete_incidents)),
    'data_versions': sorted(df['data_version'].dropna().astype(str).unique().tolist()),
    'invalid_numeric_rows': invalid_numeric_rows,
    'all_records_synthetic': bool(df['is_synthetic'].astype(bool).all())
}

print(json.dumps(validation_summary, indent=2))

assert validation_summary['measurement_rows'] == 60
assert validation_summary['distinct_source_incidents'] == 30
assert validation_summary['complete_incidents'] == 30
assert validation_summary['data_versions'] == ['1.1.0']

print('\nNotebook validation checks: PASS')


{
  "measurement_rows": 60,
  "distinct_source_incidents": 30,
  "reconstructed_incidents": 30,
  "complete_incidents": 30,
  "data_versions": [
    "1.1.0"
  ],
  "invalid_numeric_rows": 0,
  "all_records_synthetic": true
}

Notebook validation checks: PASS


## Reproducibility

The reusable scripts remain the authoritative execution path:

- `data-science/scripts/run_analytical_track.py`
- `data-science/scripts/validate_analytical_track.py`
- `data-science/scripts/export_intelligence_results.py`

This notebook provides reproducible development and validation evidence and does not introduce an alternative data source.
